# 04 · Paired bootstrap significance of the test-set results

**Purpose.** Notebook 03 reports one point estimate per configuration on the 500-question test set. This notebook
asks whether the differences between configurations are larger than test-set sampling noise.

**Method.** All six configurations were scored on the *same* questions, so every comparison is **paired**:
for each question we take the difference between two configurations, then resample those per-question differences
with replacement (10,000 times) to get a 95% percentile interval for the mean difference.

**Inputs.** Six per-question `results.parquet` files from notebook 03 (one per configuration).
**Compute.** CPU only, runs in seconds. It does not import the `adaptive_rag` package and does not re-run any model,
retrieval or generation.

**What this does not capture.** The planner was trained once (one seed). The bootstrap measures
test-set sampling variance only, not training-seed variance.

## 1 · Configuration

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json, math, zlib, itertools
import numpy as np
import pandas as pd

# ---- Where the six results live -------------------------------------------------
# Each configuration lives in its own folder written by notebook 03:
#   RESULTS_DIR / PLAN_DIRS[<config>] / "results.parquet"      (e.g. .../heldout_evaluation/test_v1_prompted/results.parquet)
# A flat file RESULTS_DIR / "results_<config>.parquet" is also accepted as a fallback.
# Colab: from google.colab import drive; drive.mount("/content/drive")
RESULTS_DIR = Path("/content/drive/MyDrive/Adaptive-RAG/results/heldout_evaluation")
PLAN_DIRS = {
    "always_once": "test_v1_always_once", "always_split": "test_v1_always_split", "prompted": "test_v1_prompted",
    "adaptive_once": "test_v1_adaptive_once", "adaptive_split": "test_v1_adaptive_split", "adaptive": "test_v1_adaptive",
}
OUT_DIR = RESULTS_DIR / "bootstrap_significance"

# ---- Bootstrap settings ---------------------------------------------------------
B = 10_000          # resamples
SEED = 544          # every comparison derives its own RNG from this seed, so results do not depend on run order
ALPHA = 0.05        # 95% interval

CONFIGS = ["always_once", "always_split", "prompted", "adaptive_once", "adaptive_split", "adaptive"]
METRICS = ["recall_at_7", "ndcg_at_7", "answer_f1", "answer_em"]   # higher is better
COST = "reranked_pairs"                                            # cost proxy: cross-encoder pairs scored per question

# Comparisons fixed in advance (first minus second). Everything else is exploratory.
PRIMARY = [
    ("adaptive",       "adaptive_once"),   # does routing help over never splitting?
    ("adaptive",       "always_once"),     # does the learned planner beat the simple default?
    ("adaptive",       "prompted"),        # does fine-tuning help over the prompted planner?
    ("always_split",   "always_once"),     # value of SPLIT, baseline planner
    ("adaptive_split", "adaptive_once"),   # value of SPLIT, fine-tuned planner
]

## 2 · Load and validate

Checks that all six files cover the same 500 questions, and counts planner failures per configuration.
A failed question has no metrics, so each comparison uses only the questions where **both** configurations succeeded.

In [ ]:
def find_results(name):
    nested = RESULTS_DIR / PLAN_DIRS[name] / "results.parquet"
    flat = RESULTS_DIR / f"results_{name}.parquet"
    for p in (nested, flat):
        if p.exists():
            return p
    raise FileNotFoundError(f"No results for '{name}'. Looked for:\n  {nested}\n  {flat}")

R = {}
for name in CONFIGS:
    df = pd.read_parquet(find_results(name))
    assert df["id"].is_unique, f"duplicate ids in {name}"
    df = df.set_index("id")
    df["ok"] = df["planner_status"].eq("ok")
    # failed questions must carry no metrics; successful ones must be complete
    assert not df.loc[df.ok, METRICS + [COST]].isna().any().any(), f"missing metrics on ok rows in {name}"
    R[name] = df

ids = R[CONFIGS[0]].index
for name in CONFIGS:
    assert set(R[name].index) == set(ids), f"{name} covers a different set of questions"
    R[name] = R[name].loc[ids]

print(f"{len(ids)} questions, sources: {R[CONFIGS[0]].source.value_counts().to_dict()}\n")
print(pd.DataFrame({
    "n_ok": {n: int(R[n].ok.sum()) for n in CONFIGS},
    "planner_failed": {n: int((~R[n].ok).sum()) for n in CONFIGS},
}).T.to_string())

## 3 · Bootstrap helpers

* `boot_means` resamples a vector of values and returns the distribution of its mean.
* `paired` compares two configurations on the questions both completed. Besides the bootstrap interval it reports
  how many questions improved, got worse or tied, and an exact sign-test p-value as a second opinion
  (recall and F1 differences are heavily tied at zero, which the sign test handles by ignoring ties).
* The bootstrap p-value is the two-sided share of resamples on the far side of zero, floored at 1/B.
* Holm adjustment is applied across the comparisons within a table, per metric.

In [ ]:
def _rng(*key):
    return np.random.default_rng(np.random.SeedSequence([SEED, zlib.crc32("|".join(map(str, key)).encode())]))

def boot_means(x, key, n_boot=B, chunk=2000):
    x = np.asarray(x, float); n = len(x); rng = _rng(*key)
    out = np.empty(n_boot)
    for s in range(0, n_boot, chunk):
        e = min(n_boot, s + chunk)
        out[s:e] = x[rng.integers(0, n, size=(e - s, n))].mean(axis=1)
    return out

def ci(bm):
    return np.percentile(bm, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])

def sign_test_p(better, worse):
    n = better + worse
    if n == 0:
        return 1.0
    k = min(better, worse)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

def paired(a, b, metric, subset=None, tag=""):
    ok = R[a].ok & R[b].ok
    if subset is not None:
        ok = ok & subset
    d = (R[a].loc[ok, metric] - R[b].loc[ok, metric]).to_numpy(float)
    bm = boot_means(d, (a, b, metric, tag))
    lo, hi = ci(bm)
    p = max(2 * min((bm <= 0).mean(), (bm >= 0).mean()), 1 / B)
    better, worse = int((d > 0).sum()), int((d < 0).sum())
    return dict(a=a, b=b, metric=metric, n=len(d), diff=d.mean(), ci_lo=lo, ci_hi=hi,
                p_boot=min(p, 1.0), better=better, worse=worse, tied=len(d) - better - worse,
                p_sign=sign_test_p(better, worse))

def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i]); adj[i] = min(1.0, running)
    return adj

def add_holm(df):
    df = df.copy(); df["p_holm"] = np.nan
    for _, g in df.groupby("metric"):
        df.loc[g.index, "p_holm"] = holm(g["p_boot"].to_numpy())
    df["ci_excludes_0"] = (df.ci_lo > 1e-9) | (df.ci_hi < -1e-9)
    return df

## 4 · Per-configuration results with 95% intervals

Each row is one configuration over the questions it completed. This reproduces the table in notebook 03,
with an interval on every number. These intervals describe each configuration on its own; use section 5 to
compare configurations, because paired differences are much tighter than overlapping marginal intervals.

In [ ]:
rows = []
for name in CONFIGS:
    ok = R[name].ok
    row = {"config": name, "n": int(ok.sum())}
    for m in METRICS + [COST]:
        x = R[name].loc[ok, m].to_numpy(float)
        lo, hi = ci(boot_means(x, (name, m, "marginal")))
        row[m] = x.mean(); row[m + "_lo"] = lo; row[m + "_hi"] = hi
    rows.append(row)
marginal = pd.DataFrame(rows)

show = marginal[["config", "n"]].copy()
for m in METRICS:
    show[m] = [f"{r[m]:.3f} [{r[m+'_lo']:.3f}, {r[m+'_hi']:.3f}]" for _, r in marginal.iterrows()]
show[COST] = [f"{r[COST]:.1f} [{r[COST+'_lo']:.1f}, {r[COST+'_hi']:.1f}]" for _, r in marginal.iterrows()]
print(show.to_string(index=False))

## 5 · Primary comparisons (fixed in advance)

Difference = first configuration minus second, so a positive recall difference favours the first.
`better / worse / tied` count questions. `p_holm` adjusts for the five primary comparisons.
For `reranked_pairs`, a positive difference means the first configuration costs more.

In [ ]:
primary = add_holm(pd.DataFrame([paired(a, b, m) for (a, b) in PRIMARY for m in METRICS + [COST]]))
primary["family"] = "primary"

def fmt(r, d=3):
    return f"{r['diff']:+.{d}f} [{r['ci_lo']:+.{d}f}, {r['ci_hi']:+.{d}f}]"

def report_table(df, metrics, pairs):
    out = []
    for (a, b) in pairs:
        row = {"comparison": f"{a} - {b}"}
        for m in metrics:
            r = df[(df.a == a) & (df.b == b) & (df.metric == m)].iloc[0]
            row[m] = fmt(r, 1 if m == COST else 3) + (" †" if r.p_holm < ALPHA else " *" if r.ci_excludes_0 else "")
        row["n"] = int(df[(df.a == a) & (df.b == b)].n.iloc[0])
        out.append(row)
    return pd.DataFrame(out)

print("mean difference [95% CI]   (* = 95% CI excludes 0, unadjusted;  † = also significant after Holm across the 5 primary comparisons)\n")
print(report_table(primary, METRICS + [COST], PRIMARY).to_string(index=False))

In [ ]:
cols = ["a", "b", "metric", "n", "diff", "ci_lo", "ci_hi", "better", "worse", "tied", "p_boot", "p_sign", "p_holm"]
pd.set_option("display.width", 220)
print(primary[primary.metric.isin(["recall_at_7", "answer_f1"])][cols].round(4).to_string(index=False))

## 6 · Exploratory: all 15 pairs on recall@7

Not fixed in advance, so read these as descriptive. Holm adjustment is applied across all 15.

In [ ]:
pairs_all = list(itertools.combinations(CONFIGS, 2))
explore = add_holm(pd.DataFrame([paired(a, b, "recall_at_7") for (a, b) in pairs_all]))
explore["family"] = "exploratory"
print(explore[["a", "b", "n", "diff", "ci_lo", "ci_hi", "better", "worse", "p_boot", "p_holm", "ci_excludes_0"]]
      .round(4).to_string(index=False))

## 7 · By source (HotpotQA vs SQuAD)

The same primary comparisons on recall@7, computed separately on each 250-question source.
Multi-hop (HotpotQA) is where SPLIT is expected to matter, so this is the most informative breakdown.

In [ ]:
by_source = []
for src in sorted(R[CONFIGS[0]].source.unique()):
    mask = R[CONFIGS[0]].source.eq(src)
    for (a, b) in PRIMARY:
        r = paired(a, b, "recall_at_7", subset=mask, tag=src); r["source"] = src
        by_source.append(r)
by_source = add_holm(pd.DataFrame(by_source)); by_source["family"] = "by_source"   # Holm across the 10 source x comparison tests
print(by_source[["source", "a", "b", "n", "diff", "ci_lo", "ci_hi", "better", "worse", "p_boot", "p_holm", "ci_excludes_0"]]
      .round(4).to_string(index=False))

## 8 · Did the learned router pick the right questions?

For each question the adaptive planner chose ONCE or SPLIT. On the questions it **routed to SPLIT**, how much did
SPLIT actually help relative to ONCE, compared with the questions it left on ONCE? If routing carried signal, the
routed questions should gain more. We also compare against **random routing** of the same number of questions
(the permutation check from notebook 03), and put a bootstrap interval on the routed-vs-unrouted gap.

In [ ]:
ad, once, split = R["adaptive"], R["adaptive_once"], R["adaptive_split"]
ok = ad.ok & once.ok & split.ok
gain = (split.loc[ok, "recall_at_7"] - once.loc[ok, "recall_at_7"])          # value of SPLIT per question
routed = ad.loc[ok, "strategy"].eq("split")

# sanity: the adaptive run must coincide with the forced runs on the questions it routed to each strategy
agree_split = np.isclose(ad.loc[ok & ad.strategy.eq("split"), "recall_at_7"], split.loc[ok & ad.strategy.eq("split"), "recall_at_7"]).mean()
agree_once = np.isclose(ad.loc[ok & ad.strategy.eq("once"), "recall_at_7"], once.loc[ok & ad.strategy.eq("once"), "recall_at_7"]).mean()
print(f"agreement with forced runs: routed->SPLIT {agree_split:.3f}, routed->ONCE {agree_once:.3f}")

g_routed, g_rest = gain[routed].to_numpy(float), gain[~routed].to_numpy(float)
bm_r, bm_u = boot_means(g_routed, ("route", "routed")), boot_means(g_rest, ("route", "rest"))
gap = bm_r - bm_u                                   # independent resamples of the two groups
obs_r, obs_u = g_routed.mean(), g_rest.mean()

rng = _rng("route", "random")
g_all = gain.to_numpy(float); m = len(g_routed)
rand_means = np.array([g_all[rng.permutation(len(g_all))[:m]].mean() for _ in range(B)])

routing = pd.DataFrame([
    dict(group=f"routed to SPLIT (n={m})", mean_gain_of_split=obs_r, ci_lo=ci(bm_r)[0], ci_hi=ci(bm_r)[1]),
    dict(group=f"left on ONCE (n={len(g_rest)})", mean_gain_of_split=obs_u, ci_lo=ci(bm_u)[0], ci_hi=ci(bm_u)[1]),
    dict(group="routed minus left-on-ONCE", mean_gain_of_split=obs_r - obs_u, ci_lo=ci(gap)[0], ci_hi=ci(gap)[1]),
])
print(routing.round(4).to_string(index=False))
print(f"\nrandom routing of {m} questions: mean gain of SPLIT = {rand_means.mean():+.4f}")
print(f"p(random routing <= observed routed gain) = {(rand_means <= obs_r).mean():.4f}")
print(f"p(random routing >= observed routed gain) = {(rand_means >= obs_r).mean():.4f}")

## 9 · Sanity check against a normal approximation

In [ ]:
# A bootstrap interval should be close to mean +/- 1.96 * SE for a mean over ~500 questions.
chk = []
for _, r in primary[primary.metric == "recall_at_7"].iterrows():
    ok_ = R[r.a].ok & R[r.b].ok
    d = (R[r.a].loc[ok_, "recall_at_7"] - R[r.b].loc[ok_, "recall_at_7"]).to_numpy(float)
    se = d.std(ddof=1) / np.sqrt(len(d))
    chk.append(max(abs(r.ci_lo - (d.mean() - 1.96 * se)), abs(r.ci_hi - (d.mean() + 1.96 * se))))
print(f"max |bootstrap endpoint - normal endpoint| over primary recall@7 comparisons: {max(chk):.4f}")

## 10 · Save

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary = pd.concat([primary, explore, by_source], ignore_index=True)
summary.to_csv(OUT_DIR / "bootstrap_paired_summary.csv", index=False)
marginal.to_csv(OUT_DIR / "bootstrap_marginal_ci.csv", index=False)
routing.to_csv(OUT_DIR / "bootstrap_routing_check.csv", index=False)
with open(OUT_DIR / "bootstrap_meta.json", "w") as f:
    json.dump(dict(B=B, seed=SEED, alpha=ALPHA, n_questions=int(len(ids)),
                   planner_failures={n: int((~R[n].ok).sum()) for n in CONFIGS},
                   primary_comparisons=[list(p) for p in PRIMARY], metrics=METRICS, cost=COST,
                   method="paired percentile bootstrap over per-question differences; Holm across comparisons within metric",
                   limitation="test-set sampling variance only; single training seed"), f, indent=2)
print("saved to", OUT_DIR)